# Yellow Sigatoka Data Preprocessing

This notebook validates the existing training and validation splits and prepares the approved model inputs. It does not alter dataset values or train a model.

## 1. Import Libraries

Use pandas and NumPy for simple data checks and input selection.

In [1]:
# Import libraries
from pathlib import Path
import numpy as np
import pandas as pd


## 2. Load Data

Load only the existing training and validation CSVs. Keep the two splits separate.

In [2]:
# Load the datasets
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets" / "yellow_sigatoka"

train_df = pd.read_csv(DATA_DIR / "YS_SIMULATED_train.csv")
validation_df = pd.read_csv(DATA_DIR / "YS_SIMULATED_validation.csv")

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)


Training shape: (4200, 32)
Validation shape: (900, 32)


## 3. Select Features

Use only the three environmental inputs from the existing score formula. The score is the target, not an input.

In [3]:
# Select model features
FEATURES = [
    "mean_temp_previous_14d_c",
    "wet_hours_previous_14d",
    "consecutive_wetness_hours",
]
TARGET = "risk_score"

feature_purpose = pd.DataFrame({
    "Feature": FEATURES,
    "Purpose": [
        "Mean temperature during the previous 14 days",
        "Estimated wet hours during the previous 14 days",
        "Current consecutive wetness duration",
    ],
})
display(feature_purpose)

# Keep the target and excluded columns out of the feature list
excluded_columns = [column for column in train_df.columns if column not in FEATURES + [TARGET]]
leakage_columns = [
    "risk_level", "temperature_suitability", "wetness_suitability",
    "bonfim_favourable_flag", "scenario_id", "record_id", "data_origin",
    "risk_score_type", "wetness_estimation_method", "label_method", "split",
    "timestamp_utc",
]
print("Target:", TARGET)
print("Excluded leakage and metadata columns present:", [c for c in leakage_columns if c in train_df.columns])


,Feature,Purpose
0,mean_temp_previous_14d_c,Mean temperature during the previous 14 days
1,wet_hours_previous_14d,Estimated wet hours during the previous 14 days
2,consecutive_wetness_hours,Current consecutive wetness duration


Target: risk_score
Excluded leakage and metadata columns present: ['risk_level', 'temperature_suitability', 'wetness_suitability', 'bonfim_favourable_flag', 'scenario_id', 'record_id', 'data_origin', 'risk_score_type', 'wetness_estimation_method', 'label_method', 'split', 'timestamp_utc']


## 4. Check Missing Values

Check required inputs and target for missing, non-numeric, NaN, and infinite values. The notebook stops if any required value is invalid; it does not fill values.

In [4]:
# Check missing values and numeric validity
required_columns = FEATURES + [TARGET]
missing_counts = pd.DataFrame({
    "Training": train_df[required_columns].isna().sum(),
    "Validation": validation_df[required_columns].isna().sum(),
})
display(missing_counts)

numeric_train = train_df[required_columns].apply(pd.to_numeric, errors="coerce")
numeric_validation = validation_df[required_columns].apply(pd.to_numeric, errors="coerce")

nonnumeric_counts = pd.DataFrame({
    "Training": (train_df[required_columns].notna() & numeric_train.isna()).sum(),
    "Validation": (validation_df[required_columns].notna() & numeric_validation.isna()).sum(),
})
nonfinite_counts = pd.DataFrame({
    "Training": pd.Series((~np.isfinite(numeric_train.to_numpy(dtype=float))).sum(), index=["All required values"]),
    "Validation": pd.Series((~np.isfinite(numeric_validation.to_numpy(dtype=float))).sum(), index=["All required values"]),
})
print("Non-numeric values:")
display(nonnumeric_counts)
print("NaN or infinite numeric values:")
display(nonfinite_counts)

if missing_counts.to_numpy().sum() or nonnumeric_counts.to_numpy().sum() or nonfinite_counts.to_numpy().sum():
    raise ValueError("Required inputs or target contain missing or invalid values. Correct the source data before continuing.")


,Training,Validation
mean_temp_previous_14d_c,0,0
wet_hours_previous_14d,0,0
consecutive_wetness_hours,0,0
risk_score,0,0


Non-numeric values:


,Training,Validation
mean_temp_previous_14d_c,0,0
wet_hours_previous_14d,0,0
consecutive_wetness_hours,0,0
risk_score,0,0


NaN or infinite numeric values:


,Training,Validation
All required values,0,0


## 5. Check Feature Ranges

Validate the wetness-hour limits, the builder's synthetic temperature range, score bounds, and the Yellow Sigatoka category thresholds.

In [5]:
# Check feature ranges and label consistency
range_rows = []
for split_name, frame in [("Training", train_df), ("Validation", validation_df)]:
    temp = numeric_train["mean_temp_previous_14d_c"] if split_name == "Training" else numeric_validation["mean_temp_previous_14d_c"]
    wet = numeric_train["wet_hours_previous_14d"] if split_name == "Training" else numeric_validation["wet_hours_previous_14d"]
    consecutive = numeric_train["consecutive_wetness_hours"] if split_name == "Training" else numeric_validation["consecutive_wetness_hours"]
    score = numeric_train[TARGET] if split_name == "Training" else numeric_validation[TARGET]
    expected_level = pd.cut(score, bins=[-np.inf, 5, 30, 65, np.inf], right=False,
                            labels=["No Risk", "Low", "Moderate", "High"]).astype(str)
    checks = [
        ("14-day mean temperature outside builder range 8 to 40 C", int((~temp.between(8, 40)).sum())),
        ("14-day wet hours outside 0 to 336", int((~wet.between(0, 336)).sum())),
        ("Consecutive wetness outside 0 to 336", int((~consecutive.between(0, 336)).sum())),
        ("Consecutive wetness above total 14-day wet hours", int((consecutive > wet).sum())),
        ("Risk score outside 0 to 100", int((~score.between(0, 100)).sum())),
        ("Risk-level labels inconsistent with score thresholds", int((expected_level != frame["risk_level"].astype(str)).sum())),
    ]
    for check, failures in checks:
        range_rows.append([split_name, check, failures])

range_checks = pd.DataFrame(range_rows, columns=["Dataset", "Check", "Invalid rows"])
display(range_checks)

if range_checks["Invalid rows"].sum() > 0:
    raise ValueError("One or more range or consistency checks failed. Source values were not changed.")


,Dataset,Check,Invalid rows
0,Training,14-day mean temperature outside builder range ...,0
1,Training,14-day wet hours outside 0 to 336,0
2,Training,Consecutive wetness outside 0 to 336,0
3,Training,Consecutive wetness above total 14-day wet hours,0
4,Training,Risk score outside 0 to 100,0
5,Training,Risk-level labels inconsistent with score thre...,0
6,Validation,14-day mean temperature outside builder range ...,0
7,Validation,14-day wet hours outside 0 to 336,0
8,Validation,Consecutive wetness outside 0 to 336,0
9,Validation,Consecutive wetness above total 14-day wet hours,0


## 6. Prepare Model Inputs

Select the requested columns in the specified order. No imputation, scaling, balancing, or feature transformation is applied.

In [6]:
# Prepare training inputs
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_validation:", X_validation.shape, "y_validation:", y_validation.shape)
print("Feature order:", list(X_train.columns))


X_train: (4200, 3) y_train: (4200,)
X_validation: (900, 3) y_validation: (900,)
Feature order: ['mean_temp_previous_14d_c', 'wet_hours_previous_14d', 'consecutive_wetness_hours']


## 7. Check Train and Validation Sets

Confirm the original row counts, matching feature order, distinct scenario IDs, and complete model inputs. Separate scenario IDs do not guarantee independent source-weather windows.

In [7]:
# Check data separation
shared_scenario_ids = set(train_df["scenario_id"].dropna()) & set(validation_df["scenario_id"].dropna())
train_duplicate_ids = int(train_df["scenario_id"].duplicated().sum())
validation_duplicate_ids = int(validation_df["scenario_id"].duplicated().sum())

separation_checks = pd.DataFrame({
    "Check": [
        "Training rows equal 4200",
        "Validation rows equal 900",
        "Feature names and order match",
        "Duplicate training scenario IDs",
        "Duplicate validation scenario IDs",
        "Shared scenario IDs",
        "Missing model inputs",
    ],
    "Result": [
        len(train_df) == 4200,
        len(validation_df) == 900,
        list(X_train.columns) == list(X_validation.columns) == FEATURES,
        train_duplicate_ids,
        validation_duplicate_ids,
        len(shared_scenario_ids),
        int(X_train.isna().sum().sum() + X_validation.isna().sum().sum()),
    ],
})
display(separation_checks)

if not separation_checks.loc[:2, "Result"].all() or any(separation_checks.loc[3:, "Result"]):
    raise ValueError("Train/validation separation checks failed. Keep the source splits unchanged and investigate.")

print("Source-weather overlap limitation: the generator permits 336-hour NASA analogue windows to overlap across scenarios.")


,Check,Result
0,Training rows equal 4200,True
1,Validation rows equal 900,True
2,Feature names and order match,True
3,Duplicate training scenario IDs,0
4,Duplicate validation scenario IDs,0
5,Shared scenario IDs,0
6,Missing model inputs,0


Source-weather overlap limitation: the generator permits 336-hour NASA analogue windows to overlap across scenarios.


## 8. Preprocessing Summary

The prepared arrays are ready for a later modeling stage. This stage does not engineer new features or train/evaluate a model. The planned backend must validate inputs and calculate rolling features from chronological measurements available at prediction time.

In [8]:
# Save the preprocessing report
from IPython.display import Markdown, display

summary_lines = [
    "# Yellow Sigatoka Preprocessing Report",
    "",
    "## Selected features and target",
    "- `mean_temp_previous_14d_c`: mean temperature during the previous 14 days.",
    "- `wet_hours_previous_14d`: estimated wet hours during the previous 14 days.",
    "- `consecutive_wetness_hours`: current consecutive wetness duration.",
    "- Target: `risk_score`.",
    "",
    "## Data validation",
    f"- Training source shape: {train_df.shape}; validation source shape: {validation_df.shape}.",
    f"- Prepared shapes: X_train {X_train.shape}, y_train {y_train.shape}, X_validation {X_validation.shape}, y_validation {y_validation.shape}.",
    f"- Missing required cells: training {int(missing_counts['Training'].sum())}; validation {int(missing_counts['Validation'].sum())}.",
    f"- Non-numeric required cells: training {int(nonnumeric_counts['Training'].sum())}; validation {int(nonnumeric_counts['Validation'].sum())}.",
    f"- Non-finite required values: training {int(nonfinite_counts['Training'].iloc[0])}; validation {int(nonfinite_counts['Validation'].iloc[0])}.",
    "- All required-value, range, target-bound, and risk-level consistency checks passed.",
    "",
    "## Range checks",
    "- Both wetness-hour features are within 0 to 336 hours.",
    "- Consecutive wetness does not exceed total wet hours in the 14-day window.",
    "- 14-day mean temperature is within the builder's synthetic range of 8 to 40 C; this is not a new biological cutoff.",
    "- Risk scores are within 0 to 100 and labels match No Risk <5, Low 5 to <30, Moderate 30 to <65, and High >=65.",
    "",
    "## Excluded columns and preprocessing decisions",
    "The target is not an input. Risk category, temperature/wetness suitability components, Bonfim favourable flag, record/scenario IDs, source fields, label-generation fields, timestamp, and split metadata are excluded. Other weather fields are not added automatically.",
    "No rows were resplit, shuffled, balanced, or combined. No missing values were filled, no features were scaled or transformed, and no model was trained or evaluated.",
    "",
    "## Train/validation separation and limitation",
    f"- Duplicate scenario IDs: training {train_duplicate_ids}; validation {validation_duplicate_ids}.",
    f"- Shared scenario IDs: {len(shared_scenario_ids)}.",
    "- Scenario IDs are disjoint, but the builder allows overlapping 336-hour NASA analogue source-weather windows. Full source independence is not established.",
    "",
    "## Future backend input requirements",
    "A future FastAPI request should validate finite numeric values for `mean_temp_previous_14d_c` (8 to 40 C), `wet_hours_previous_14d` (0 to 336), and `consecutive_wetness_hours` (0 to 336 and no greater than 14-day wet hours). The backend should calculate these rolling features from chronological environmental readings available at prediction time. The current simulated records are not evidence of deployed IoT sensor measurements.",
    "",
    "## Next step",
    "Proceed to the separate feature engineering review. Keep the three formula inputs distinct from any added candidate features, and discuss the deterministic synthetic target and source-window overlap before model training.",
]
report_path = PROJECT_ROOT / "reports" / "yellow_sigatoka" / "preprocessing" / "Preprocessing_Report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("Saved report:", report_path)
display(Markdown("Preprocessing validation passed. No model was trained or evaluated."))


Saved report: C:\Users\HP\Desktop\ML training\reports\yellow_sigatoka\preprocessing\Preprocessing_Report.md


Preprocessing validation passed. No model was trained or evaluated.